# 01_cache — Cache as a layer over the pipeline

Cache is neither business logic (the meaning does not change whether a result was computed or served from cache) nor passive infrastructure (it changes the **execution path** — a result can return without running the pipeline at all). AOA splits two concerns: the **operation** decides *meaning* (which key to build, whether to store a given result and under which tags, what to invalidate), the **coordinator** decides *policy* (where to store, how long to live, eviction).

This notebook caches a "heavy" config request and serves its repeat from cache; a "light" request is not cached, so its repeat always runs the pipeline.

**What's new**

| Concept | Description |
|---------|-------------|
| `CacheCoordinator()` | Opt-in cache wired on the machine |
| `cache_key(params)` | Builds the user key segment (`None` → skip cache); MUST include scope (`tenant_id`) |
| `CacheTag(type, key)` | Typed tag for indexing entries; at least one field non-`None` |
| `on_cache_write(result, params, duration_ms)` | Returns `list[CacheTag]` (store + index) or `None` (skip) |
| cache hit | the pipeline does **not** run — no aspect executes |
| `on_cache_invalidate(params, result)` | Evicts tagged entries after every clean pipeline |
| (also `read_cache`) | freshness instead of TTL — return `None` to invalidate |

In [ ]:
!pip install aoa-action-machine

In [ ]:
import asyncio

from pydantic import Field

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.logging import Channel, ConsoleLogger
from aoa.action_machine.logging.log_coordinator import LogCoordinator
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.action_machine.runtime.cache_coordinator import CacheCoordinator
from aoa.action_machine.runtime.cache_tag import CacheTag

# Cache only when the pipeline took at least this long (ms)
HEAVY_MIN_MS = 50.0

## Domain, Params, Result

In [ ]:
class RootDomain(BaseDomain):
    name = "root"
    description = "Root domain"


class ConfigParams(BaseParams):
    tenant_id: str = Field(description="Tenant ID")
    name: str = Field(description="Config name")


class ConfigResult(BaseResult):
    value: str = Field(description="Config value")

## The action — `cache_key` (with scope) + `on_cache_write`

`cache_key` builds the key from `tenant_id:name` — the `tenant_id` scope is mandatory, otherwise one tenant could be served another's cached answer. `on_cache_write` returns `list[CacheTag] | None`: `None` skips the write; a list stores the result and indexes it under those tags. Here only results that took at least `HEAVY_MIN_MS` to compute are cached.

In [ ]:
@meta(description="Get config (cached)", domain=RootDomain)
@check_roles(GuestRole)
class GetConfigAction(BaseAction[ConfigParams, ConfigResult]):

    def cache_key(self, params: ConfigParams) -> str | None:
        return f"{params.tenant_id}:{params.name}"

    async def on_cache_write(
        self,
        result: ConfigResult,
        params: ConfigParams,
        duration_ms: float,
    ) -> list[CacheTag] | None:
        if duration_ms < HEAVY_MIN_MS:
            print(f"  on_cache_write: duration_ms={duration_ms:.1f}, write=no (too light)")
            return None
        print(f"  on_cache_write: duration_ms={duration_ms:.1f}, write=yes")
        return [CacheTag(key=f"{params.tenant_id}:{params.name}")]

    @summary_aspect("Load config")
    async def load_summary(self, params, state, box, connections):
        heavy = params.name != "ping"
        if heavy:
            await asyncio.sleep(0.08)
        await box.info(
            Channel.business,
            "{%var.kind} request: tenant={%var.tenant}, name={%var.name}",
            kind="heavy" if heavy else "light",
            tenant=params.tenant_id,
            name=params.name,
        )
        return ConfigResult(value=f"{params.tenant_id}:{params.name}:value")

## Run

The "heavy again" block is empty — that is the cache hit: the machine returned the stored result without running a single aspect. The "light" request is never stored, so its repeat runs the pipeline again. (`duration_ms` numbers vary per run.)

> In Colab, `await` works at top level — no `asyncio.run()`.

In [ ]:
async def main() -> None:
    machine = ActionProductMachine(
        log_coordinator=LogCoordinator(loggers=[ConsoleLogger()]),
        cache_coordinator=CacheCoordinator(),
    )

    print("Sample cache — heavy request\n")
    params = ConfigParams(tenant_id="acme", name="feature-flags")
    await machine.run(Context(), GetConfigAction(), params=params)

    print("\nSample cache — heavy again (cached)\n")
    await machine.run(Context(), GetConfigAction(), params=params)

    print("\nSample cache — light request\n")
    await machine.run(
        Context(),
        GetConfigAction(),
        params=ConfigParams(tenant_id="acme", name="ping"),
    )

    print("\nSample cache — light again (not in cache)\n")
    await machine.run(
        Context(),
        GetConfigAction(),
        params=ConfigParams(tenant_id="acme", name="ping"),
    )


await main()